# 7.4 · 注册中心、SDK 分级与路线图

**The Registry, SDK tiers and the roadmap**

> **学完本课你能：** 编写并校验 `server.json`，在经过验证的命名空间下将其发布到 MCP 注册中心（MCP Registry），像聚合器一样消费注册中心，并讲清 SDK 分级（SDK tier）、SEP 流程以及路线图的优先方向。

官方阅读材料：[The MCP Registry](https://modelcontextprotocol.io/registry/about) · [Registry quickstart](https://modelcontextprotocol.io/registry/quickstart) · [SDK tiering](https://modelcontextprotocol.io/community/sdk-tiers) · [Roadmap](https://modelcontextprotocol.io/development/roadmap)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 编写 `server.json` 并按模式校验
下面是官方 `2025-12-11` 版 `server.schema.json` 的节选。约束条件抄自已发布的模式（Schema），只保留了我们用到的字段。我们先用 `jsonschema` 校验，再加上注册中心自己的规则：不得使用版本范围、版本不得为 `latest`、命名空间必须与认证方式匹配，以及发布者元数据不得超过 4 KB 的限制。

In [ ]:
import json

import jsonschema

SCHEMA_EXCERPT = {
    "type": "object",
    "required": ["name", "description", "version"],
    "properties": {
        "$schema": {"type": "string"},
        "name": {"type": "string", "minLength": 3, "maxLength": 200, "pattern": r"^[a-zA-Z0-9.-]+/[a-zA-Z0-9._-]+$"},
        "title": {"type": "string", "minLength": 1, "maxLength": 100},
        "description": {"type": "string", "minLength": 1, "maxLength": 100},
        "version": {"type": "string", "maxLength": 255},
        "repository": {"type": "object", "required": ["url", "source"]},
        "packages": {"type": "array", "items": {
            "type": "object", "required": ["registryType", "identifier", "transport"],
            "properties": {
                "registryType": {"type": "string"}, "identifier": {"type": "string"},
                "version": {"type": "string", "minLength": 1, "not": {"const": "latest"}},
                "fileSha256": {"type": "string", "pattern": "^[a-f0-9]{64}$"},
                "transport": {"type": "object", "required": ["type"],
                              "properties": {"type": {"enum": ["stdio", "streamable-http", "sse"]}}}}}},
        "remotes": {"type": "array", "items": {
            "type": "object", "required": ["type", "url"],
            "properties": {"type": {"enum": ["streamable-http", "sse"]},
                           "url": {"type": "string", "pattern": r"^https?://[^\s]+$"},
                           "variables": {"type": "object"}}}},
    },
}

server_json = {
    "$schema": "https://static.modelcontextprotocol.io/schemas/2025-12-11/server.schema.json",
    "name": "io.github.my-username/weather",
    "title": "Weather",
    "description": "An MCP server for weather information.",
    "repository": {"url": "https://github.com/my-username/mcp-weather-server", "source": "github"},
    "version": "1.0.1",
    "packages": [{"registryType": "npm", "identifier": "@my-username/mcp-weather-server", "version": "1.0.1",
                  "transport": {"type": "stdio"}}],
    "remotes": [{"type": "streamable-http", "url": "https://{tenant_id}.weather.example.com/mcp",
                 "variables": {"tenant_id": {"description": "Your tenant identifier", "isRequired": True}}}],
}

RANGE_MARKERS = ("^", "~", ">", "<", "||", " - ", ".x", ".*", "*")


def registry_checks(doc: dict, auth: dict) -> list[str]:
    problems = [f"schema: {e.message} at /{'/'.join(map(str, e.path))}"
                for e in jsonschema.Draft7Validator(SCHEMA_EXCERPT).iter_errors(doc)]
    versions = [doc.get("version", "")] + [p.get("version", "") for p in doc.get("packages", [])]
    for v in versions:
        if any(m in v for m in RANGE_MARKERS) or v.endswith(".x"):
            problems.append(f"version range prohibited: {v!r}")
    namespace = doc.get("name", "").split("/")[0]
    if auth["method"] == "github" and namespace.lower() != f"io.github.{auth['login']}".lower():
        problems.append(f"GitHub auth only allows io.github.{auth['login']}/*, got {namespace}/*")
    if auth["method"] in ("dns", "http"):
        domain_ns = ".".join(reversed(auth["domain"].split(".")))
        if not (namespace == domain_ns or namespace.startswith(domain_ns + ".")):
            problems.append(f"{auth['domain']} only allows {domain_ns}.*/*, got {namespace}/*")
    publisher = doc.get("_meta", {}).get("io.modelcontextprotocol.registry/publisher-provided")
    if publisher is not None and len(json.dumps(publisher).encode()) > 4096:
        problems.append("publisher-provided _meta exceeds 4096 bytes")
    return problems


print("valid?", registry_checks(server_json, {"method": "github", "login": "my-username"}) or "yes")

### 2. 故意把它弄坏
下面每一个错误都出现在文档的故障排查表或模式（Schema）中。它们都应当在你运行 `mcp-publisher publish` 之前被发现。

In [ ]:
import copy

mistakes = {
    "caret range":            lambda d: d.update(version="^1.0.1"),
    "package 'latest'":       lambda d: d["packages"][0].update(version="latest"),
    "two slashes in name":    lambda d: d.update(name="io.github.my-username/weather/v2"),
    "description too long":   lambda d: d.update(description="An MCP server for weather " * 5),
    "wrong namespace":        lambda d: d.update(name="io.github.someone-else/weather"),
    "remote not http(s)":     lambda d: d["remotes"][0].update(url="ftp://weather.example.com/mcp"),
    "missing transport":      lambda d: d["packages"][0].pop("transport"),
    "huge publisher _meta":   lambda d: d.update(_meta={"io.modelcontextprotocol.registry/publisher-provided":
                                                        {"blob": "x" * 5000}}),
}
for label, mutate in mistakes.items():
    doc = copy.deepcopy(server_json)
    mutate(doc)
    found = registry_checks(doc, {"method": "github", "login": "my-username"})
    print(f"{label:<22} -> {found[0] if found else 'NOT CAUGHT'}")

print("\nDNS auth for example.com:",
      registry_checks({**server_json, "name": "com.example.tools/weather"}, {"method": "dns", "domain": "example.com"}) or "ok")

### 3. 按软件包类型进行归属验证
注册中心（Registry）会检查每个软件包是否反过来指向服务器名称。这个模拟器遵循文档中的规则。注意 Cargo 的陷阱：crates.io 会剥除 HTML 注释，因此隐藏的 `<!-- mcp-name: … -->` 在 PyPI 上能通过，在 Cargo 上却会失败。

In [ ]:
import hashlib
import re

NAME = "io.github.username/widget-mcp"


def verify_ownership(registry_type: str, artifact: dict, name: str = NAME) -> bool:
    if registry_type == "npm":
        return artifact["package.json"].get("mcpName") == name
    if registry_type in ("pypi", "nuget"):
        return f"mcp-name: {name}" in artifact["README"]
    if registry_type == "cargo":   # rendered README: comments are stripped before the validator looks
        rendered = re.sub(r"<!--.*?-->", "", artifact["README"], flags=re.S)
        return f"mcp-name: {name}" in rendered
    if registry_type == "oci":
        return artifact["labels"].get("io.modelcontextprotocol.server.name") == name
    if registry_type == "mcpb":
        sha_ok = hashlib.sha256(artifact["bytes"]).hexdigest() == artifact["fileSha256"]
        return "mcp" in artifact["identifier"] and sha_ok   # (the registry skips the hash; CLIENTS check it)
    raise ValueError(registry_type)


hidden = f"# Widget\n\n<!-- mcp-name: {NAME} -->\n"
visible = f"# Widget\n\n- MCP Registry name: `mcp-name: {NAME}`\n"
bundle = b"fake mcpb bytes"
cases = [
    ("npm", {"package.json": {"name": "@username/widget-mcp", "mcpName": NAME}}),
    ("pypi", {"README": hidden}),
    ("cargo", {"README": hidden}),
    ("cargo", {"README": visible}),
    ("oci", {"labels": {"io.modelcontextprotocol.server.name": NAME}}),
    ("mcpb", {"identifier": "https://github.com/username/widget-mcp/releases/download/v1.0.0/widget.mcpb",
              "bytes": bundle, "fileSha256": hashlib.sha256(bundle).hexdigest()}),
    ("mcpb", {"identifier": "https://github.com/username/widget-mcp/releases/download/v1.0.0/widget.mcpb",
              "bytes": bundle + b"tampered", "fileSha256": hashlib.sha256(bundle).hexdigest()}),
]
for registry_type, artifact in cases:
    print(f"{registry_type:<6} {'hidden comment' if artifact.get('README') == hidden else '':<15} ->",
          "verified" if verify_ownership(registry_type, artifact) else "REJECTED")

### 4. 按聚合器应有的方式对版本排序
这是一个小型 semver 解析器（核心版本号加上预发布版本的优先级规则），结合文档中的四步比较法：先看 latest 标记，再看 semver，再看发布时间戳，最后是 semver 胜过非 semver。示例来自版本管理页面。这正是聚合器（Aggregator）应当采用的排序方式。

In [ ]:
import functools

SEMVER = re.compile(r"^(0|[1-9]\d*)\.(0|[1-9]\d*)\.(0|[1-9]\d*)(?:-([0-9A-Za-z.-]+))?(?:\+[0-9A-Za-z.-]+)?$")


def parse_semver(v: str):
    m = SEMVER.match(v)
    if not m:
        return None
    core = tuple(int(x) for x in m.groups()[:3])
    pre = m.group(4)
    ids = tuple((0, int(p), "") if p.isdigit() else (1, 0, p) for p in pre.split(".")) if pre else None
    return core, ids


def semver_cmp(a, b) -> int:
    (ca, pa), (cb, pb) = a, b
    if ca != cb:
        return -1 if ca < cb else 1
    if pa == pb:
        return 0
    if pa is None:          # a release sorts AFTER any of its prereleases
        return 1
    if pb is None:
        return -1
    return -1 if pa < pb else 1


def aggregator_cmp(x: dict, y: dict) -> int:
    """Negative if x is older than y. x/y: {'version', 'isLatest', 'publishedAt'}."""
    if x.get("isLatest") != y.get("isLatest"):
        return 1 if x.get("isLatest") else -1
    sx, sy = parse_semver(x["version"]), parse_semver(y["version"])
    if sx and sy:
        return semver_cmp(sx, sy)
    if not sx and not sy:
        return (x["publishedAt"] > y["publishedAt"]) - (x["publishedAt"] < y["publishedAt"])
    return 1 if sx else -1


for v in ["1.0.0", "2.1.3-alpha", "1.0.0-beta.1", "3.0.0-rc.2", "2025.11.25", "2025.6.18", "2025.06.18", "2025-06-18", "v1.0"]:
    print(f"{v:<13} semver={'yes' if parse_semver(v) else 'no '}")

history = [{"version": "1.2.3", "publishedAt": "2026-01-01"}, {"version": "1.2.3-1", "publishedAt": "2026-02-01"},
           {"version": "1.10.0", "publishedAt": "2026-03-01"}, {"version": "1.9.0", "publishedAt": "2026-04-01"}]
ordered = sorted(history, key=functools.cmp_to_key(aggregator_cmp))
print("\noldest -> newest:", [h["version"] for h in ordered])
print("note: 1.2.3-1 was published AFTER 1.2.3 but sorts before it; 1.10.0 > 1.9.0 numerically, not lexically")
print("non-semver vs semver:", aggregator_cmp({"version": "2025-06-18", "publishedAt": "2026-05-01"},
                                                {"version": "0.0.1", "publishedAt": "2020-01-01"}), "(negative = older)")

### 5. 运行在 127.0.0.1 上的本地注册中心 API，并像聚合器一样消费它
我们以<strong>与线上 API 相同的响应结构</strong>提供一个小型数据集。每一项都是 `{"server": <server.json>, "_meta": {"io.modelcontextprotocol.registry/official": {status, publishedAt, updatedAt, isLatest}}}`，列表则携带 `metadata.nextCursor` / `count`。文档中的三个端点都已实现，支持 `search`、`limit`/`cursor` 分页和 `updated_since`。然后一个聚合器（Aggregator）分页遍历全部数据，跟踪 `status`，并对名称进行 URL 编码。

In [ ]:
import logging
import socket
import threading
import time
from urllib.parse import quote, unquote

import httpx
import uvicorn
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route

logging.getLogger("httpx").setLevel(logging.WARNING)
OFFICIAL = "io.modelcontextprotocol.registry/official"


def record(name, version, published, *, latest=False, status="active", remotes=None, desc="Demo server"):
    server = {"$schema": "https://static.modelcontextprotocol.io/schemas/2025-12-11/server.schema.json",
              "name": name, "description": desc, "version": version}
    if remotes:
        server["remotes"] = [{"type": "streamable-http", "url": remotes}]
    else:
        server["packages"] = [{"registryType": "npm", "identifier": "@demo/" + name.split("/")[1],
                               "version": version, "transport": {"type": "stdio"}}]
    return {"server": server, "_meta": {OFFICIAL: {"status": status, "publishedAt": published,
                                                    "updatedAt": published, "isLatest": latest}}}


DATA = [record("io.github.alice/weather", "1.0.0", "2026-05-01T10:00:00Z"),
        record("io.github.alice/weather", "1.1.0", "2026-06-01T10:00:00Z", latest=True),
        record("com.example/acme-analytics", "2.0.0", "2026-06-15T09:00:00Z", latest=True,
               remotes="https://analytics.example.com/mcp", desc="Business intelligence and reporting"),
        record("io.github.bob/spam-server", "0.0.1", "2026-07-01T00:00:00Z", latest=True, status="deleted"),
        record("io.modelcontextprotocol/everything", "2026.7.28", "2026-07-28T12:00:00Z", latest=True,
               desc="Reference server exercising every MCP feature"),
        record("io.github.carol/weather-alerts", "0.3.0", "2026-08-02T08:00:00Z", latest=True)]
DATA.sort(key=lambda r: (r["server"]["name"], r["server"]["version"]))


async def list_servers(request: Request):
    q = request.query_params
    items = DATA
    if search := q.get("search"):
        items = [r for r in items if search.lower() in r["server"]["name"].lower()]
    if since := q.get("updated_since"):
        items = [r for r in items if r["_meta"][OFFICIAL]["updatedAt"] > since]
    if cursor := q.get("cursor"):
        keys = [f"{r['server']['name']}:{r['server']['version']}" for r in items]
        items = items[keys.index(cursor) + 1:] if cursor in keys else []
    limit = int(q.get("limit", 30))
    page = items[:limit]
    meta = {"count": len(page)}
    if len(items) > limit:
        meta["nextCursor"] = f"{page[-1]['server']['name']}:{page[-1]['server']['version']}"
    return JSONResponse({"servers": page, "metadata": meta})


async def list_versions(request: Request):
    name = unquote(request.path_params["server_name"])
    rows = [r for r in DATA if r["server"]["name"] == name]
    return JSONResponse({"servers": rows, "metadata": {"count": len(rows)}}, status_code=200 if rows else 404)


async def get_version(request: Request):
    name, version = unquote(request.path_params["server_name"]), unquote(request.path_params["version"])
    for r in DATA:
        meta = r["_meta"][OFFICIAL]
        if r["server"]["name"] == name and (r["server"]["version"] == version or (version == "latest" and meta["isLatest"])):
            return JSONResponse(r)
    return JSONResponse({"error": "not found"}, status_code=404)


app = Starlette(routes=[Route("/v0.1/servers", list_servers),
                        Route("/v0.1/servers/{server_name:path}/versions", list_versions),
                        Route("/v0.1/servers/{server_name:path}/versions/{version}", get_version)])
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
registry = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
threading.Thread(target=registry.run, daemon=True).start()
while not registry.started:
    time.sleep(0.05)
BASE = f"http://127.0.0.1:{port}"
print("local registry at", BASE)

In [ ]:
http = httpx.Client(base_url=BASE, trust_env=False)

# 1) aggregator full sync with cursor pagination
mirror, cursor, pages = {}, None, 0
while True:
    params = {"limit": 2} | ({"cursor": cursor} if cursor else {})
    body = http.get("/v0.1/servers", params=params).json()
    pages += 1
    for item in body["servers"]:
        mirror[(item["server"]["name"], item["server"]["version"])] = item
    cursor = body["metadata"].get("nextCursor")
    if not cursor:
        break
print(f"synced {len(mirror)} records in {pages} pages")

# 2) drop deleted servers, keep the latest version per name
visible = {}
for (name, _), item in mirror.items():
    official = item["_meta"][OFFICIAL]
    if official["status"] == "deleted":
        continue
    if official["isLatest"]:
        visible[name] = item["server"]["version"]
print("catalogue (latest only, deleted removed):", json.dumps(visible, indent=1))

# 3) search, incremental sync, and URL-encoded per-server endpoints
print("\nsearch=weather ->", [i["server"]["name"] + "@" + i["server"]["version"]
                              for i in http.get("/v0.1/servers", params={"search": "weather"}).json()["servers"]])
print("updated_since=2026-07-01 ->", [i["server"]["name"] for i in
                                      http.get("/v0.1/servers", params={"updated_since": "2026-07-01T00:00:00Z"}).json()["servers"]])
encoded = quote("io.modelcontextprotocol/everything", safe="")
print("GET /v0.1/servers/" + encoded + "/versions/latest ->",
      http.get(f"/v0.1/servers/{encoded}/versions/latest").json()["server"]["version"])
print("versions of alice/weather ->", [i["server"]["version"] for i in
                                       http.get(f"/v0.1/servers/{quote('io.github.alice/weather', safe='')}/versions").json()["servers"]])

### 6. 子注册中心添加自己的 `_meta`
子注册中心（Subregistry）可以在一个以自身命名的键下注入自定义数据。官方的 `_meta` 条目保持不变，服务器自身的字段也是不可变的。

In [ ]:
def enrich(item: dict, ratings: dict) -> dict:
    out = copy.deepcopy(item)
    name = out["server"]["name"]
    out["_meta"]["com.example.subregistry/custom"] = {
        "user_rating": ratings.get(name, {}).get("rating"),
        "download_count": ratings.get(name, {}).get("downloads", 0),
        "security_scan": {"last_scanned": "2026-10-01T12:00:00Z", "vulnerabilities_found": 0}}
    return out


ratings = {"io.github.alice/weather": {"rating": 4.5, "downloads": 12345}}
latest_weather = http.get(f"/v0.1/servers/{quote('io.github.alice/weather', safe='')}/versions/latest").json()
print(json.dumps(enrich(latest_weather, ratings)["_meta"], indent=2))
http.close()
registry.should_exit = True

### 7. SDK 分级自评与 SEP 状态机
分级表被写成一个函数。把一个 SDK 的各项指标传进去，它就会报告该 SDK 达到的最高级别（SDK tier），以及阻碍它升到下一级的因素。SEP 工作流是一个小型状态机：检查哪些状态转换是合法的，以及 `final` 需要满足什么条件。

In [ ]:
TIERS = {
    1: {"conformance": 100, "triage_days": 2, "p0_days": 7, "feature_lag_months": 0,
        "stable_release": "clear-versioning", "docs": "comprehensive", "dependency_policy": True, "roadmap": "published"},
    2: {"conformance": 80, "triage_days": 30, "p0_days": 14, "feature_lag_months": 6,
        "stable_release": "at-least-one", "docs": "basic", "dependency_policy": True, "roadmap": "plan"},
}
RANK = {"none": 0, "basic": 1, "comprehensive": 2, "at-least-one": 1, "clear-versioning": 2, "plan": 1, "published": 2}


def assess(sdk: dict) -> tuple[int, list[str]]:
    for tier in (1, 2):
        req, gaps = TIERS[tier], []
        if sdk["conformance"] < req["conformance"]:
            gaps.append(f"conformance {sdk['conformance']}% < {req['conformance']}%")
        if sdk["triage_days"] > req["triage_days"]:
            gaps.append(f"triage {sdk['triage_days']}d > {req['triage_days']}d")
        if sdk["p0_days"] > req["p0_days"]:
            gaps.append(f"P0 fix {sdk['p0_days']}d > {req['p0_days']}d")
        if sdk["feature_lag_months"] > req["feature_lag_months"]:
            gaps.append(f"features lag {sdk['feature_lag_months']} months")
        for key in ("stable_release", "docs", "roadmap"):
            if RANK[sdk[key]] < RANK[req[key]]:
                gaps.append(f"{key}: {sdk[key]} (needs {req[key]})")
        if req["dependency_policy"] and not sdk["dependency_policy"]:
            gaps.append("no published dependency policy")
        if tier == 1:
            tier1_gaps = gaps
        if not gaps:
            return tier, tier1_gaps   # tier met; tier1_gaps = what still blocks Tier 1 (empty for Tier 1)
    return 3, tier1_gaps


my_sdk = {"conformance": 93, "triage_days": 1, "p0_days": 5, "feature_lag_months": 2,
          "stable_release": "clear-versioning", "docs": "comprehensive", "dependency_policy": True, "roadmap": "published"}
tier, gaps = assess(my_sdk)
print(f"my SDK: Tier {tier}; blocking Tier 1: {gaps or 'nothing'}")
print("perfect SDK:", assess({**my_sdk, "conformance": 100, "feature_lag_months": 0})[0])
print("weekend project:", assess({"conformance": 40, "triage_days": 90, "p0_days": 60, "feature_lag_months": 12,
                                  "stable_release": "none", "docs": "none", "dependency_policy": False, "roadmap": "none"}))

SEP_FLOW = {"awaiting-sponsor": {"draft", "dormant", "withdrawn"}, "draft": {"in-review", "withdrawn"},
            "in-review": {"accepted", "rejected", "draft"}, "accepted": {"final", "superseded"},
            "dormant": {"draft"}, "rejected": set(), "withdrawn": set(), "final": {"superseded"}}


def walk(path, *, sep_type="Standards Track", observable=True, reference_impl=True, conformance_merged=True):
    for a, b in zip(path, path[1:]):
        if b not in SEP_FLOW[a]:
            return f"illegal transition {a} -> {b}"
        if b == "final":
            if not reference_impl:
                return "cannot be final: reference implementation missing"
            if sep_type == "Standards Track" and observable and not conformance_merged:
                return "cannot be final: conformance scenario + traceability file not merged"
    return "ok: " + " -> ".join(path)


print("\n" + walk(["awaiting-sponsor", "draft", "in-review", "accepted", "final"]))
print(walk(["awaiting-sponsor", "draft", "in-review", "accepted", "final"], conformance_merged=False))
print(walk(["awaiting-sponsor", "draft", "in-review", "accepted", "final"], sep_type="Extensions Track",
           conformance_merged=False))
print(walk(["awaiting-sponsor", "dormant", "draft"]), "(dormant can be revived)")
print(walk(["awaiting-sponsor", "accepted"]))

### 动手练习
1. 把 `server_json` 改成 `com.example/acme-analytics` 名下的一个<strong>纯远程</strong>服务器，同时提供一个 `streamable-http` 远程端点和一个已弃用的 `sse` 远程端点，外加一个机密的 `X-API-Key` 请求头。用针对 `example.com` 的 DNS 认证对它进行校验。
2. 编写 `resolve_remote_url(remote, values)`。它应当替换 `{tenant_id}` 风格的变量，应用 `default` 默认值，强制执行 `choices` 和 `isRequired`，并检查结果仍然符合模式中的 URL 正则。
3. 扩展本地注册中心，让 `PATCH` 风格的状态变更（`active` → `deprecated`）会更新 `updatedAt`。然后展示一个使用 `updated_since` 同步的聚合器无需完整重新抓取即可获取这一变更。
4. 用 `assess()` 找出让 `my_sdk` 升到 Tier 1 所需的最小改动。然后模拟降级：如果最新稳定版本上有一个一致性测试开始失败，并且持续四周未修复，会发生什么？